# DeepSeek-V4-Flash-0731 on a single AMD MI308X (gfx942 / 80 CU)Reproducible vLLM + ROCm serving and benchmark walkthrough:- 512K configured context (500K validated)- DSpark K=7 speculative decoding- prefix caching + native CPU-KV tierRun the cells top to bottom. Prerequisites: clone `deepseek-v4-flash-mi308x`, place the pinned vLLM/AITER/flydsl wheels, and install the runtime (see the repository README quick start steps 0-3). Path defaults are overridable via environment variables.

## 1. MI308X / ROCm environment detection

In [ ]:
!rocminfo | grep -E 'Name:|Marketing Name:|Compute Unit' | head -20!rocm-smi --showproductname --showmeminfo vramimport torchprint('torch:', torch.__version__)print('cuda available:', torch.cuda.is_available())print('device:', torch.cuda.get_device_name(0))print('gfx arch:', torch.cuda.get_device_properties(0).gcnArchName)

## 2. Runtime versions

In [ ]:
import importlib.metadata as mdfor pkg in ('torch', 'transformers', 'vllm', 'aiter', 'flydsl', 'triton', 'flash_attn'):    try:        print(f'{pkg}: {md.version(pkg)}')    except Exception as exc:        print(f'{pkg}: <not found> ({exc})')

## 3. Model download (ModelScope first, verifies 48/48 shards)

In [ ]:
!bash scripts/01_download_model.sh dsflash

## 4. Launch the OpenAI-compatible server (background)The stable profile is `--max-model-len 524288 --max-num-batched-tokens 3072`, DSpark K=7, 16 GB pinned GPU KV + 12 GB native CPU tier. Audit the runtime before serving.

In [ ]:
!python3 scripts/audit_runtime.py!nohup bash scripts/02_serve_vllm.sh dsflash > serve.log 2>&1 &print('server launching in background; watch serve.log')

## 5. Health check (poll until ready)

In [ ]:
import time, requestsbase = 'http://127.0.0.1:8000'for attempt in range(120):    try:        resp = requests.get(base + '/health', timeout=5)        if resp.status_code == 200:            print('ready:', resp.json())            break    except Exception:        pass    time.sleep(5)else:    print('server did not become ready; check serve.log')

## 6. Chat completions test

In [ ]:
import requests, osbase = os.environ.get('VLLM_BASE_URL', 'http://127.0.0.1:8000')resp = requests.post(    base + '/v1/chat/completions',    json={        'model': 'deepseek-v4-flash',        'messages': [{'role': 'user', 'content': '一句话解释 MI308X 的 gfx942 架构。'}],        'max_tokens': 256,    },    timeout=300,)print('status:', resp.status_code)print(resp.json()['choices'][0]['message']['content'])

## 7. Performance test (agent trace + fixed decode)

In [ ]:
!python3 scripts/bench/bench_agent_trace.py 30 20000!python3 scripts/04_bench_decode.py 512

## 8. VRAM usage and teardown

In [ ]:
!rocm-smi --showmeminfo vram!pkill -f 'bin/vllm serve' && echo 'server stopped'